In [25]:
# import google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# import necessary packages
import pandas as pd

In [43]:
# seasons dictionary
regions_and_seasons_dict = {
    'south_sudan': ['MJJ', 'JAS', 'ASO'],
    'eastern_east_africa': ['OND', 'MAM'],
    'lake_victoria_basin': ['DJF', 'MAM', 'SON'],
    'west_africa': ['JAS'],
    'southern_africa': ['DJF', 'FMA'],
    'south_sudan': ['MJJ', 'JAS', 'ASO'],
    'eastern_ukraine': ['DJF', 'AMJ', 'JA'],
    'sri_lanka': ['OND']
}

In [44]:
def pivot_seasonal_csv(file_path, regions_and_seasons_dict, save_path):
  """This function takes a file path to a csv file preprocessed with
  the convert_monthly_to_seasonal.py script, and pivots the seasonal portion.

  the csv file has the following columns:
  - time
  - lead_time
  - latitude
  - longitude
  - predicted_precip
  - precip
  - year
  - month
  - season columns (e.g., MJJ, JAS, OND, etc.)

  This function assumes that there is an index column
  that is not needed, and as a result, index_col=0.

  output csv file has the following columns:
  - time
  - lead_time
  - latitude
  - longitude
  - predicted_precip
  - precip
  - year
  - month
  - lead_category (short, medium, long)
  - season (MJJ, JAS, OND, etc.)

  The output csv drops all NaN values to solve the overlapping seasons
  problem present in the original data processed by convert_monthly_to_seasonal.py

  The saved csv will have naming convention
  some_region_model_merged_seasonal_pivoted.csv

  Arguments
  ----------
  file_path : str
      path to csv file; make sure that the format follows:
      some_system_path/data/seasonal/eastern_east_africa_CCSM4_merged_seasonal.csv
      It does not matter what the name of your data folder is, just that the
      csv file has the name of some_region_model_merged_seasonal.csv

  regions_and_seasons_dict : dict
      dictionary of regions and seasons, refer to the above dictionary definition

  save_path : str
      path to save pivoted csv file; make sure that the format follows:
      some_system_path/data/seasonal/
      Make sure there is a slash (/) at the end of the path
  """

  df = pd.read_csv(file_path, index_col=0)

  # extract the year and month from time
  df['year'] = pd.DatetimeIndex(df['time']).year
  df['month'] = pd.DatetimeIndex(df['time']).month

  split_path = file_path.split('/')
  file_name = split_path[-1]
  current_region = '_'.join(file_name.split('_')[0:-3])
  current_model = file_name.split('_')[-3]

  print(f"Currently Processing {current_region} for {current_model} model")

  seasons = regions_and_seasons_dict[current_region]
  season_dfs = []

  for season in seasons:
      if season in df.columns:
          temp_df = df[['time', 'lead_time', 'latitude', 'longitude',
                                'predicted_precip', 'precip', 'year', 'month', season]].copy()
          temp_df = temp_df.rename(columns={season: 'lead_category'})
          temp_df = temp_df.dropna(subset=['lead_category'])  # Drop rows where lead_category is NaN
          temp_df['season'] = season
          season_dfs.append(temp_df)

  # Combine all seasonal rows
  long_format_df = pd.concat(season_dfs, ignore_index=True)

  # Keep only the last part after splitting by underscore (i.e, OND_short -> short)
  long_format_df['lead_category'] = long_format_df['lead_category'].str.split('_').str[-1]

  # assign model name column
  long_format_df['model'] = str(current_model)

  # save as csv
  long_format_df.to_csv(save_path + file_name.replace('_merged_seasonal.csv', '_merged_seasonal_pivoted.csv'), index=False)

  return long_format_df

In [50]:
# run pivot_seasonal on all files in csv/seasonal
import os
import glob

# list all files in the csv/seasonal path
files = glob.glob('/content/drive/MyDrive/capstone_data/csv/seasonal/*.csv')

# iterate through each file
for f in files:
  # Normalize path to use forward slashes
  f = f.replace('\\', '/')
  pivot_seasonal_csv(f, regions_and_seasons_dict=regions_and_seasons_dict, save_path='/content/drive/MyDrive/capstone_data/csv/seasonal/')


In [65]:
# open pivoted data
pivoted_data_eea = pd.read_csv('/content/drive/MyDrive/capstone_data/csv/seasonal/sri_lanka_CanESM5_merged_seasonal_pivoted.csv')

In [66]:
# take seasonal means
pivoted_data_eea = pivoted_data_eea.drop(['time', 'lead_time', 'month'], axis=1).groupby(['latitude', 'longitude', 'season', 'lead_category', 'model', 'year'])[['predicted_precip', 'precip']].mean().reset_index()

In [63]:
pivoted_data_eea

,latitude,longitude,season,lead_category,model,year,predicted_precip,precip
0,6.0,80.5,OND,long,CanESM5,1991,8.631926,271.964787
1,6.0,80.5,OND,long,CanESM5,1992,7.941944,229.142187
2,6.0,80.5,OND,long,CanESM5,1993,8.394635,297.126257
3,6.0,80.5,OND,long,CanESM5,1994,8.959564,292.376163
4,6.0,80.5,OND,long,CanESM5,1995,7.819471,171.644237
...,...,...,...,...,...,...,...,...
2203,9.5,80.5,OND,short,CanESM5,2017,4.810624,204.774127
2204,9.5,80.5,OND,short,CanESM5,2018,6.435378,268.083793
2205,9.5,80.5,OND,short,CanESM5,2019,6.296166,280.294330
2206,9.5,80.5,OND,short,CanESM5,2020,6.276733,291.488667
